In [1]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = PROJECT_ROOT / "data" / "step4"

articles = pd.read_csv(DATA / "articles.csv", encoding="utf-8-sig")
failures = pd.read_csv(DATA / "failed_urls.csv", encoding="utf-8-sig")
stats_label = pd.read_csv(DATA / "stats_by_label.csv", encoding="utf-8-sig")
stats_source = pd.read_csv(DATA / "stats_by_source.csv", encoding="utf-8-sig")
same_title = pd.read_csv(DATA / "same_title_review.csv", encoding="utf-8-sig")
content_dups = pd.read_csv(DATA / "duplicate_content_review.csv", encoding="utf-8-sig")
manifest = json.loads((DATA / "manifest.json").read_text(encoding="utf-8"))

print({k: manifest[k] for k in ["total_candidates", "success", "failed", "success_rate_pct"]})

{'total_candidates': 6000, 'success': 5364, 'failed': 185, 'success_rate_pct': 89.4}


In [2]:
required = ["ID", "Title", "Content", "URL", "Source",
            "Published_Date", "Original_Category", "Label"]
assert all(c in articles.columns for c in required)
assert articles.ID.nunique() == len(articles)
assert articles.URL.nunique() == len(articles)
assert articles.Content_Length.min() >= 300
assert set(articles.Label) == {"Kinh tế", "Thế giới", "Thể thao", "Pháp luật",
                               "Giáo dục", "Sức khỏe", "Công nghệ", "Du lịch"}
stats_label

,Label,Target,Success,Failed,Not_Attempted,Success_Rate_Pct
0,Công nghệ,750,647,3,100,86.27
1,Du lịch,750,647,3,100,86.27
2,Giáo dục,750,633,17,100,84.40
3,Kinh tế,750,740,10,0,98.67
4,Pháp luật,750,653,46,51,87.07
5,Sức khỏe,750,647,3,100,86.27
6,Thế giới,750,738,12,0,98.40
7,Thể thao,750,659,91,0,87.87


In [3]:
stats_source

,Source,Target,Success,Failed,Not_Attempted,Success_Rate_Pct
0,DanTri,800,232,117,451,29.00
1,ThanhNien,400,398,2,0,99.50
2,TuoiTre,400,399,1,0,99.75
3,VietnamNet,3200,3164,36,0,98.88
4,VnExpress,1200,1171,29,0,97.58


In [4]:
print("Độ dài nội dung:")
print(articles.Content_Length.describe().round(1).to_string())
print("\nThiếu ngày đăng:", int(articles.Published_Date.isna().sum()))
print("Thiếu tác giả:", int(articles.Author.isna().sum()))
print("Số bài:", len(articles))
articles.sample(6, random_state=42)[
    ["ID", "Source", "Label", "Content_Length", "Published_Date", "Author", "Title"]
]

Độ dài nội dung:
count     5364.0
mean      3300.9
std       1681.8
min        395.0
25%       2093.0
50%       3017.0
75%       4177.8
max      34096.0

Thiếu ngày đăng: 32
Thiếu tác giả: 947
Số bài: 5364


,ID,Source,Label,Content_Length,Published_Date,Author,Title
5061,C005696,VietnamNet,Du lịch,3387,2026-10-23T06:00:00+07:00,Thảo Trinh,9X Hà Nội ghé bản làng quanh năm xanh mát ở Tu...
1557,C001581,VnExpress,Thể thao,4295,2026-11-16T08:07:36+07:00,VnExpress,Tuchel: '30 phút cuối trước Tây Ban Nha dài nh...
3922,C004353,VietnamNet,Sức khỏe,3717,2026-10-30T14:56:00+07:00,Nhật An,5 loại cây mọc hoang quanh vườn hỗ trợ ổn định...
1103,C001126,VietnamNet,Thế giới,2292,2026-11-18T16:27:00+07:00,Tuấn Anh,"Ông Trump lên đường tới Trung Quốc, chuẩn bị c..."
4568,C005102,VietnamNet,Công nghệ,3300,2026-11-02T09:20:00+07:00,Du Lam,Nam diễn viên chuyên đóng 'tổng tài' về quê bá...
401,C000405,VietnamNet,Kinh tế,1988,2026-11-17T15:45:00+07:00,VietNamNet News,Thưởng trà Phúc Long theo cách hoàn toàn mới


In [5]:
failures.head(20)

,ID,URL,Source,Label,Original_Category,HTTP_Status,Error,Attempts,Last_Attempt,Crawl_Worker
0,C000100,https://vnexpress.net/lam-gi-de-tranh-bi-ngop-...,VnExpress,Kinh tế,Kinh doanh,200.0,parse: nội dung quá ngắn (150 ký tự),1,2026-11-20T15:22:59+07:00,VnExpress-w0
1,C000137,https://vnexpress.net/ba-nguyen-van-linh-thua-...,VnExpress,Kinh tế,Kinh doanh,200.0,parse: không tìm thấy khối nội dung,1,2026-11-20T15:22:59+07:00,VnExpress-w1
2,C000212,https://thanhnien.vn/norton-park-chuan-song-ti...,ThanhNien,Kinh tế,Kinh tế,200.0,parse: nội dung quá ngắn (0 ký tự),1,2026-11-20T15:22:59+07:00,ThanhNien-w0
3,C000435,https://vietnamnet.vn/30-nam-chung-khoan-viet-...,VietnamNet,Kinh tế,Kinh doanh,200.0,parse: nội dung quá ngắn (0 ký tự),1,2026-11-20T15:22:59+07:00,VietnamNet-w0
4,C000459,https://vietnamnet.vn/6-4-trieu-ty-dong-von-da...,VietnamNet,Kinh tế,Kinh doanh,200.0,parse: nội dung quá ngắn (0 ký tự),1,2026-11-20T15:22:59+07:00,VietnamNet-w1
5,C000480,https://vietnamnet.vn/xang-dau-don-dap-giam-gi...,VietnamNet,Kinh tế,Kinh doanh,200.0,parse: nội dung quá ngắn (0 ký tự),1,2026-11-20T15:22:59+07:00,VietnamNet-w2
6,C000645,https://vietnamnet.vn/gia-dien-tang-giam-nhu-t...,VietnamNet,Kinh tế,Kinh doanh,200.0,parse: nội dung quá ngắn (0 ký tự),1,2026-11-20T15:23:00+07:00,VietnamNet-w0
7,C000676,https://vietnamnet.vn/nhieu-loai-thue-ha-ve-0-...,VietnamNet,Kinh tế,Kinh doanh,200.0,parse: nội dung quá ngắn (0 ký tự),1,2026-11-20T15:23:00+07:00,VietnamNet-w1
8,C000711,https://vietnamnet.vn/gia-vang-roi-lien-tiep-n...,VietnamNet,Kinh tế,Kinh doanh,200.0,parse: nội dung quá ngắn (0 ký tự),1,2026-11-20T15:23:00+07:00,VietnamNet-w2
9,C000720,https://vietnamnet.vn/gia-bac-boc-hoi-gan-80-t...,VietnamNet,Kinh tế,Kinh doanh,200.0,parse: nội dung quá ngắn (0 ký tự),1,2026-11-20T15:23:02+07:00,VietnamNet-w0


In [6]:
not_attempted = pd.read_csv(DATA / "not_attempted_urls.csv", encoding="utf-8-sig")
print("Chưa thử:", len(not_attempted))
print(not_attempted.groupby("Source").size().to_string())
not_attempted.head(10)

Chưa thử: 451
Source
DanTri    451


,ID,URL,Source,Label,Original_Category,Reason
0,C002547,https://dantri.com.vn/phap-luat/dai-gia-dieu-c...,DanTri,Pháp luật,Pháp luật,DanTri bị ConnectTimeout/HTTP 428; dừng worker...
1,C002549,https://dantri.com.vn/phap-luat/nhung-man-gia-...,DanTri,Pháp luật,Pháp luật,DanTri bị ConnectTimeout/HTTP 428; dừng worker...
2,C002551,https://dantri.com.vn/phap-luat/hot-girl-ly-me...,DanTri,Pháp luật,Pháp luật,DanTri bị ConnectTimeout/HTTP 428; dừng worker...
3,C002553,https://dantri.com.vn/phap-luat/tam-giu-ke-cam...,DanTri,Pháp luật,Pháp luật,DanTri bị ConnectTimeout/HTTP 428; dừng worker...
4,C002554,https://dantri.com.vn/phap-luat/tam-giu-nghi-p...,DanTri,Pháp luật,Pháp luật,DanTri bị ConnectTimeout/HTTP 428; dừng worker...
5,C002555,https://dantri.com.vn/phap-luat/thu-giu-gan-17...,DanTri,Pháp luật,Pháp luật,DanTri bị ConnectTimeout/HTTP 428; dừng worker...
6,C002556,https://dantri.com.vn/phap-luat/cuop-cap-nhan-...,DanTri,Pháp luật,Pháp luật,DanTri bị ConnectTimeout/HTTP 428; dừng worker...
7,C002557,https://dantri.com.vn/phap-luat/bat-qua-tang-n...,DanTri,Pháp luật,Pháp luật,DanTri bị ConnectTimeout/HTTP 428; dừng worker...
8,C002558,https://dantri.com.vn/phap-luat/can-tho-xu-pha...,DanTri,Pháp luật,Pháp luật,DanTri bị ConnectTimeout/HTTP 428; dừng worker...
9,C002559,https://dantri.com.vn/phap-luat/cho-pitbull-so...,DanTri,Pháp luật,Pháp luật,DanTri bị ConnectTimeout/HTTP 428; dừng worker...


In [7]:
print("Nhóm trùng tiêu đề:", same_title.Title_Duplicate_Group.nunique() if len(same_title) else 0)
print("Nhóm nội dung giống hệt:", content_dups.Content_Duplicate_Group.nunique() if len(content_dups) else 0)
print("Số bài vẫn được giữ:", len(articles))
same_title.head(20)

Nhóm trùng tiêu đề: 4
Nhóm nội dung giống hệt: 1
Số bài vẫn được giữ: 5364


,Title_Duplicate_Group,Content_Relation,ID,Source,Label,Title,URL,Content_Hash
0,TITLE00001,different_content,C001085,DanTri,Thế giới,Tổng thống Mông Cổ thông báo từ chức,https://dantri.com.vn/the-gioi/tong-thong-mong...,8869e3e85e28181bf3dc6162dcf77dbe78b31df40b60fd...
1,TITLE00001,different_content,C000847,VnExpress,Thế giới,Tổng thống Mông Cổ thông báo từ chức,https://vnexpress.net/tong-thong-mong-co-thong...,8afa429a9736330076ec646a257470536356f691ff1b0b...
2,TITLE00002,different_content,C000215,ThanhNien,Kinh tế,Vingroup khởi công dự án nhà ở xã hội lớn nhất...,https://thanhnien.vn/vingroup-khoi-cong-du-an-...,f9396f9f145fcfdc1293793f96dde802912546f3e5209e...
3,TITLE00002,different_content,C000235,ThanhNien,Kinh tế,Vingroup khởi công dự án nhà ở xã hội lớn nhất...,https://thanhnien.vn/vingroup-khoi-cong-du-an-...,6500023b0c1f69a68f913183ee84a98401a4f10aa3b470...
4,TITLE00003,different_content,C003232,ThanhNien,Giáo dục,Thứ trưởng Bộ GD-ĐT nói về 'điều rất đẹp ở ngh...,https://thanhnien.vn/thu-truong-bo-gd-dt-noi-v...,1e587ece7e998c9a82ab7f51ad9ec5cf946c6a610baa96...
5,TITLE00003,different_content,C003370,VietnamNet,Giáo dục,Thứ trưởng Bộ GD-ĐT nói về điều rất đẹp ở nghề...,https://vietnamnet.vn/thu-truong-bo-gd-dt-noi-...,9f88b950cafe4fe0fe7fc788bb39687b28eb07ff91dd55...
6,TITLE00004,different_content,C002681,VietnamNet,Pháp luật,Giang hồ mạng ‘Bình Trọc’ bị khởi tố,https://vietnamnet.vn/giang-ho-mang-binh-troc-...,59143ac152caa661009db34420dad0eef96c363de00509...
7,TITLE00004,different_content,C002376,VnExpress,Pháp luật,Giang hồ mạng 'Bình trọc' bị khởi tố,https://vnexpress.net/giang-ho-mang-binh-troc-...,f12b0555d52ca0436217982f3ce523f3df608f682911da...
